# Hyperparameter tuning

Weights are learned. Hyperparameters, such as depth, width and learning rate, are chosen by you. Too many to set by hand, and they interact, so we search. This notebook writes a seeded random search by hand, then the same with KerasTuner, and ends on the main trap: the winning score is optimistic.

## Why this matters

The wrong learning rate can leave a good model stuck or diverging, so a model is only as good as the knobs you set. Tuning picks them well instead of by guesswork.

Picture a search with a fixed budget. Each trial is a full training run, and you can afford only a handful. You spend that budget to find the setting that works, so where you look matters more than how hard you look.

Two calculations shape the search. The learning rate lives on a log scale, so sample it log-uniform: draw log10(eta) uniformly and set eta to 10**uniform(-4, -1), which spreads trials evenly across 1e-4 to 1e-1 instead of bunching near the top. And the winner's curse: the best of many noisy scores is optimistic, because picking the minimum rewards luck as well as skill, so report the final model on a held-out test set no trial ever saw.

Order: intuition, maths, from scratch, library, tuning, what breaks.

In [ ]:
import tempfile

from common.dl import set_seed, setup_keras

setup_keras("torch")
import keras
import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from common.datasets import load_california_housing

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=3, suppress=True)

## 1. Intuition

Tuning is a search with an expensive scoring function: every candidate costs a full training run. With a budget of nine runs, how should you spend them?

A grid uses three values per knob, every combination. Random search picks each knob independently. Both use nine runs, but only one tries nine values of the knob that matters.

## 2. Maths and cost function

Score each candidate $c$ by validation loss $V(c)$ and keep the best. The cost function for tuning is that validation loss, never the test loss.

A grid with $k$ values per knob across 2 knobs tries $k^2$ runs but only $k$ distinct values of each. Random search tries $k^2$ distinct values of each. When one knob matters, as is usual, it explores that knob $k$ times as finely for the same cost.

A learning rate lives on a log scale, so sample $\log_{10}\eta$ uniformly.

## 3. From scratch

California housing, 6,000 rows, scaled. A candidate is a depth, a width and a learning rate. The search loop uses a seeded generator, so the same search runs each time.

In [ ]:
# Data: California housing, 6,000 rows, 8 features, scaled to zero mean and unit variance.
Xh, yh = load_california_housing(6000, seed=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(Xh.to_numpy(), yh.to_numpy(), random_state=SEED)
scaler = StandardScaler().fit(X_tr)
X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)

# A trial is one (depth, width, learning rate). The grid repeats 3 learning rates;
# random search draws each rate log-uniform via 10**uniform(-4, -1) because the rate
# spans decades (1e-4 to 1e-1), so random search explores the knob that matters more
# finely. rng is seeded with SEED so the same search runs every time.
grid_lr = np.repeat([1e-3, 1e-2, 1e-1], 3)
rng = np.random.default_rng(SEED)
rand_lr = 10 ** rng.uniform(-4, -1, 9)
print("distinct learning rates, grid of 9:", len(set(grid_lr)), "| random of 9:", len(set(rand_lr)))


def build(depth: int, width: int, lr: float) -> keras.Model:
    model = keras.Sequential([keras.Input((8,))])
    for _ in range(depth):
        model.add(keras.layers.Dense(width, activation="relu"))
    model.add(keras.layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="mse")
    return model


def train_one(depth: int, width: int, lr: float, seed: int, epochs: int = 20) -> float:
    set_seed(seed)
    hist = build(depth, width, lr).fit(
        X_tr, y_tr, epochs=epochs, batch_size=512, validation_split=0.2, verbose=0
    )
    return float(hist.history["val_loss"][-1])


rng = np.random.default_rng(SEED)
trials = []
for i in range(8):
    cfg = {
        "depth": int(rng.integers(1, 4)),
        "width": int(rng.choice([8, 16, 32, 64, 128])),
        "lr": float(10 ** rng.uniform(-4, -1)),
    }
    trials.append({**cfg, "val_mse": train_one(**cfg, seed=SEED)})
results = pd.DataFrame(trials).sort_values("val_mse").reset_index(drop=True)
print(results.round(4).to_string())

The top three trials all sit at learning rates of 0.01 to 0.03. The worst use 0.0005 to 0.002, barely trained in 20 epochs (validation MSE of 1.1 and 1.9, against 0.35 for the best). The learning rate dominates, where random search beats a grid.

Plot the same trials against the learning rate on a log axis to see the pattern directly.

In [ ]:
def plot_trial_scatter(
    trials: "pd.DataFrame",
    x: str = "lr",
    score: str = "val_mse",
    ax: "matplotlib.axes.Axes | None" = None,
) -> "matplotlib.axes.Axes":
    """Scatter pre-computed trial scores against a hyperparameter.

    Reads the already-run trials DataFrame and plots trials[x] against trials[score].
    The x axis is log-scaled because the learning rate spans decades. No randomness:
    nothing is sampled or trained here, the function only draws existing numbers.
    """
    if ax is None:
        _, ax = plt.subplots()
    ax.scatter(trials[x], trials[score])
    ax.set_xscale("log")
    ax.set_xlabel(x)
    ax.set_ylabel(score)
    ax.set_title(f"{score} against {x}")
    return ax


plot_trial_scatter(results)
plt.show()

## 4. Library version

KerasTuner wraps the loop: describe the space in a `build(hp)` function, and it runs the trials, tracks and ranks them. It installs with `uv sync --extra torch` on the Keras 3 torch backend. One note: version 1.4 imports `six` without declaring it, so the project lists `six` explicitly. Its `RandomSearch` takes the same space and budget as ours.

In [ ]:
def build_hp(hp: kt.HyperParameters) -> keras.Model:
    return build(
        depth=hp.Int("depth", 1, 3),
        width=hp.Choice("width", [8, 16, 32, 64, 128]),
        lr=hp.Float("lr", 1e-4, 1e-1, sampling="log"),
    )


tuner = kt.RandomSearch(
    build_hp,
    objective="val_loss",
    max_trials=6,
    seed=SEED,
    directory=tempfile.mkdtemp(),
    project_name="housing",
    overwrite=True,
)
tuner.search(X_tr, y_tr, epochs=20, batch_size=512, validation_split=0.2, verbose=0)
best = tuner.get_best_hyperparameters(1)[0]
print("best:", best.values)
print("best val loss:", round(tuner.oracle.get_best_trials(1)[0].score, 4))

Candidates came from the library's own random stream, so the trials differ from ours, but the story is the same. Its value is bookkeeping, plus strategies (`Hyperband`, `BayesianOptimization`) that drop bad trials early.

## 5. Tuning

Refine around the winner of the hand-written search: keep its depth and width, and try five learning rates from 0.01 to 0.1.

In [ ]:
top = results.iloc[0]
print(
    f"hand search winner: depth {int(top.depth)}, width {int(top.width)}, lr {top.lr:.4f}, val {top.val_mse:.3f}"
)
for lr in (0.01, 0.02, 0.04, 0.07, 0.1):
    print(f"  lr {lr:<5}: val MSE {train_one(int(top.depth), int(top.width), lr, SEED):.3f}")

Between 0.01 and 0.04 the score moves by 0.016, noise-sized. At 0.07 and above it jumps to 1.36, a cliff. The winner sat in a flat region with a sharp edge a little further up. The pattern is standard: wide and cheap first, then a narrower round.

## 6. What breaks

The best of eight scores is selected because it is lowest, and part of that is luck from the random start. Re-run the winning configuration with three other seeds.

In [ ]:
cfg = {"depth": int(top.depth), "width": int(top.width), "lr": float(top.lr)}
reruns = [train_one(**cfg, seed=s) for s in (1, 2, 3)]
print(f"selected score {top.val_mse:.3f}, same config on other seeds: {np.round(reruns, 3)}")
print(f"median of all 8 trials: {results.val_mse.median():.3f}")

All three re-runs are worse than the selected score of 0.351 (0.360 to 0.371). Picking the minimum of many noisy scores flatters the winner (the winner's curse), and more trials make it larger. Report the final model on a test set no trial ever saw, and treat differences smaller than the seed-to-seed spread as ties.